In [2]:
import datasets
import os
import pandas as pd
import json
from create_datasets import get_label_set
import numpy as np

cwd = os.getcwd()

In [3]:
letter = "a"
htype = "mhead"
#ds = datasets.load_from_disk(f"{cwd}/inputs/{letter}/sent/{htype}_ds")
ds = datasets.load_from_disk(f"{cwd}/inputs/{letter}/{htype}_dsdcts/dsdct_r2")

In [14]:
entry = 6
lbls = ['labels_Actor', 'labels_InstrumentType', 'labels_Objective', 'labels_Resource', 'labels_Time']
for i in range(len(ds[entry]['tokens'])):
    #print(f"{ds[entry]['tokens'][i]} {ds[entry]['ner_tags'][i]}")
    string = f"{ds[entry]['tokens'][i]} "
    for lbl in lbls:
        string+= f"{ds[entry][lbl][i]} "
    print(string)

it O O O O O 
shall O O O O O 
not O O O O O 
affect O O O O O 
the O O O O O 
validity O O O O O 
of O O O O O 
any O O O O O 
delegated O B O O O 
acts O I O O O 
already O O O O O 
in O O O O O 
force O O O O O 
. O O O O O 


start with the a dataset

In [2]:
letter = "a"
htype = "mhead"
ds = datasets.load_from_disk(f"{cwd}/inputs/{letter}/{htype}_ds")
label_lst = get_label_set(letter, htype)

get total spans&tokens, per-label spans&tokens, and ratios

In [3]:
total_counts = {label: {"spans":0,"tokens":0} for label in label_lst}
total_spans = 0
total_labeled_tokens = 0
for row in ds:
    for label in label_lst:
        spans = row[f"labels_{label}"].count("B")
        innies = row[f"labels_{label}"].count("I")
        total_counts[label]["spans"] += spans
        total_counts[label]["tokens"] += spans+innies
        #
        total_spans+= spans
        total_labeled_tokens+=spans+innies
print(f"Total Spans: {total_spans}; Total (Labeled) Tokens: {total_labeled_tokens}")
print(total_counts)
for label in list(total_counts):
    print(f"{label}: {round((total_counts[label]['spans']/total_spans)*100,2)}% all spans, {round((total_counts[label]['tokens']/total_labeled_tokens)*100,2)}% all (labeled) tokens")

Total Spans: 10706; Total (Labeled) Tokens: 31694
{'Actor': {'spans': 5381, 'tokens': 11838}, 'InstrumentType': {'spans': 3145, 'tokens': 7426}, 'Objective': {'spans': 942, 'tokens': 7991}, 'Resource': {'spans': 481, 'tokens': 1217}, 'Time': {'spans': 757, 'tokens': 3222}}
Actor: 50.26% all spans, 37.35% all (labeled) tokens
InstrumentType: 29.38% all spans, 23.43% all (labeled) tokens
Objective: 8.8% all spans, 25.21% all (labeled) tokens
Resource: 4.49% all spans, 3.84% all (labeled) tokens
Time: 7.07% all spans, 10.17% all (labeled) tokens


In [ ]:
#weights
wghts_a_1 = {}
#wghts_a_2 = {}
for lbl in list(total_counts):
    #print(f"{lbl}: {total_spans/(total_counts[lbl]['spans']*len(total_counts))}")
    wghts_a_1[lbl]= total_spans/(total_counts[lbl]['spans']*len(total_counts))
    #wghts_a_2[lbl]= 1-total_counts[lbl]['spans']/total_spans
wghts_a_1#, wghts_a_2

({'Actor': 0.39791860249024347,
  'InstrumentType': 0.6808267090620032,
  'Objective': 2.273036093418259,
  'Resource': 4.451559251559251,
  'Time': 2.828533685601057},
 {'Actor': 0.4973846441247899,
  'InstrumentType': 0.7062394918737156,
  'Objective': 0.9120119559125723,
  'Resource': 0.9550719222865682,
  'Time': 0.9292919858023538})

for each of the 412 articles, get label counts and total percentages for spans&tokens

In [104]:
tracking = {}
for row in ds:
    tracking[row['id']] = {label: {"spans":0,"tokens":0} for label in label_lst}
    for label in label_lst:
        spans = row[f"labels_{label}"].count("B")
        innies = row[f"labels_{label}"].count("I")
        tracking[row['id']][label]["spans"] = spans
        tracking[row['id']][label]["tokens"] = spans+innies
        #
        tracking[row['id']][label]["span_pct"] = round((spans/total_counts[label]['spans'])*100,3)
        tracking[row['id']][label]["token_pct"] = round(((spans+innies)/total_counts[label]['tokens'])*100,3)

for each label, gathering the articles where they occur

In [106]:
arts_of_occurrence = {label:[] for label in label_lst}
for artid in list(tracking):
    for label in label_lst:
        spct = tracking[artid][label]["span_pct"]
        if spct>0:
            #arts_of_occurrence[label].append((artid, spct))
            arts_of_occurrence[label].append(artid)
for label in list(arts_of_occurrence):
    print(f"{label}: {len(arts_of_occurrence[label])}")

Actor: 371
InstrumentType: 312
Objective: 204
Resource: 135
Time: 246


find a way to dynamically create a threshold of inclusion of minority classes

In [122]:
lbl_arts_occ = []
min_obj = (None,100)
max_obj = (None,0)
for label in list(arts_of_occurrence):
    val = round((len(arts_of_occurrence[label])/412)*100,2)
    lbl_arts_occ.append((label,val))
    print(f"{label}: {val}")
    if val<min_obj[1]:
        min_obj = (label,val)
    if val>max_obj[1]:
        max_obj = (label,val)
#for label in list(arts_of_occurrence):
#    if len(arts_of_occurrence[label])/412<0.5:
#        minority_labels.append(label)
vals = [entry[1] for entry in lbl_arts_occ]
thresh = np.quantile(vals, .333)
print(f"Thresh: {thresh}")
minority_labels = []
for entry in lbl_arts_occ:
    if entry[1]<thresh:
        minority_labels.append(entry[0])
print(minority_labels)
print(min_obj, max_obj)
min_label = min_obj[0]
max_label = max_obj[0]

Actor: 90.05
InstrumentType: 75.73
Objective: 49.51
Resource: 32.77
Time: 59.71
Thresh: 52.8964
['Objective', 'Resource']
('Resource', 32.77) ('Actor', 90.05)


building a new list of articles based on lowest minority classes

In [123]:
arts_of_interest = []
for label in minority_labels:
    arts_of_interest.extend(arts_of_occurrence[label])
arts_of_interest = list(set(arts_of_interest))
print(len(arts_of_interest))

240


some method of sorting/reducing the number of articles to oversample from
for now, sorting descending by pct of most minority label span
then taking the first half of that list

In [129]:
aoi_dict = dict()
for key in arts_of_interest:
    aoi_dict[key] = tracking[key]
# sort by lowest minority label
#arts_inorder = sorted(aoi_dict, key=lambda x:aoi_dict[x][min_label]['span_pct'], reverse=True)
arts_dscmin = sorted(aoi_dict, key=lambda x:aoi_dict[x][min_label]['span_pct'], reverse=True)
arts_ascmax = sorted(aoi_dict, key=lambda x:aoi_dict[x][max_label]['span_pct'], reverse=False)
#aoi_dict_sorted = {key: aoi_dict[key] for key in arts_inorder}
aoi_dict_dscmin = {key: aoi_dict[key] for key in arts_dscmin}
aoi_dict_ascmax = {key: aoi_dict[key] for key in arts_ascmax}
# may need to find a way of reducing the number of articles dynamically?
# for now just take the first half
#overs_arts = arts_inorder[:int(len(arts_inorder)/2)]
overs_dscmin = arts_dscmin[:int(len(arts_dscmin)/2)]
overs_ascmax = arts_ascmax[:int(len(arts_ascmax)/2)]
overs_arts = list(set(overs_dscmin+overs_ascmax))

make a new dataset, the oversampling dataset, using the new articles
then concatenate them

In [130]:
osds = ds.filter(lambda example: example["id"] in overs_arts)
ovs_ds = datasets.concatenate_datasets([ds, osds])
ovs_ds

Filter: 100%|██████████| 412/412 [00:00<00:00, 880.73 examples/s]


Dataset({
    features: ['id', 'text', 'tokens', 'labels_Actor', 'labels_InstrumentType', 'labels_Objective', 'labels_Resource', 'labels_Time'],
    num_rows: 605
})

In [131]:
new_counts = {label: {"spans":0,"tokens":0} for label in label_lst}
new_spans = 0
new_labeled_tokens = 0
for row in ovs_ds:
    for label in label_lst:
        spans = row[f"labels_{label}"].count("B")
        innies = row[f"labels_{label}"].count("I")
        new_counts[label]["spans"] += spans
        new_counts[label]["tokens"] += spans+innies
        #
        new_spans+= spans
        new_labeled_tokens+=spans+innies
print(f"Total Spans: {new_spans}; Total (Labeled) Tokens: {new_labeled_tokens}")
print(new_counts)
for label in list(new_counts):
    print(f"{label}: {round((new_counts[label]['spans']/new_spans)*100,2)}% all spans, {round((new_counts[label]['tokens']/new_labeled_tokens)*100,2)}% all (labeled) tokens")

Total Spans: 17472; Total (Labeled) Tokens: 52958
{'Actor': {'spans': 8467, 'tokens': 18924}, 'InstrumentType': {'spans': 5155, 'tokens': 12362}, 'Objective': {'spans': 1709, 'tokens': 14492}, 'Resource': {'spans': 955, 'tokens': 2420}, 'Time': {'spans': 1186, 'tokens': 4760}}
Actor: 48.46% all spans, 35.73% all (labeled) tokens
InstrumentType: 29.5% all spans, 23.34% all (labeled) tokens
Objective: 9.78% all spans, 27.37% all (labeled) tokens
Resource: 5.47% all spans, 4.57% all (labeled) tokens
Time: 6.79% all spans, 8.99% all (labeled) tokens


In [158]:
def get_overall_counts(ds, label_lst):
    # initialize values
    total_counts = {label: {"spans":0,"tokens":0} for label in label_lst}
    total_spans = 0
    total_labeled_tokens = 0
    # get counts for label spans and tokens
    for row in ds:
        for label in label_lst:
            spans = row[f"labels_{label}"].count("B")
            innies = row[f"labels_{label}"].count("I")
            total_counts[label]["spans"] += spans
            total_counts[label]["tokens"] += spans+innies
            total_spans+= spans
            total_labeled_tokens+=spans+innies
    total_counts["Overall"] = {"spans":total_spans,"tokens":total_labeled_tokens}
    for label in list(total_counts):
        print(f"{label}: {round((total_counts[label]['spans']/total_spans)*100,2)}% all spans ({total_counts[label]['spans']})")
    return total_counts

def article_label_coverage(ds, label_lst, total_counts):
    tracking = {}
    for row in ds:
        tracking[row['id']] = {label: {"spans":0,"tokens":0} for label in label_lst}
        for label in label_lst:
            spans = row[f"labels_{label}"].count("B")
            innies = row[f"labels_{label}"].count("I")
            tracking[row['id']][label]["spans"] = spans
            tracking[row['id']][label]["tokens"] = spans+innies
            #
            tracking[row['id']][label]["span_pct"] = round((spans/total_counts[label]['spans'])*100,3)
            tracking[row['id']][label]["token_pct"] = round(((spans+innies)/total_counts[label]['tokens'])*100,3)
    return tracking

def get_art_lists_for_lbls(art_lbl_tracking, label_lst):
    arts_of_occurrence = {label:[] for label in label_lst}
    for artid in list(art_lbl_tracking):
        for label in label_lst:
            spct = art_lbl_tracking[artid][label]["span_pct"]
            if spct>0:
                arts_of_occurrence[label].append(artid)
    return arts_of_occurrence

def compare_art_lbl_occ(arts_of_occurrence, quant=0.333):
    lbl_arts_occ = []
    min_obj = (None,100)
    max_obj = (None,0)
    for label in list(arts_of_occurrence):
        val = round((len(arts_of_occurrence[label])/412)*100,2)
        lbl_arts_occ.append((label,val))
        #print(f"{label}: {val}")
        # get min and max label names
        if val<min_obj[1]:
            min_obj = (label,val)
        if val>max_obj[1]:
            max_obj = (label,val)
    # get threshold by finding quantile value
    vals = [entry[1] for entry in lbl_arts_occ]
    thresh = np.quantile(vals, quant)
    # determine minority labels
    minority_labels = []
    for entry in lbl_arts_occ:
        if entry[1]<thresh:
            minority_labels.append(entry[0])
    min_label = min_obj[0]
    max_label = max_obj[0]
    arts_of_interest = []
    for label in minority_labels:
        arts_of_interest.extend(arts_of_occurrence[label])
    arts_of_interest = list(set(arts_of_interest))
    print("Minority Labels:",minority_labels)
    return arts_of_interest, minority_labels, min_label, max_label

def compare_overall_lbl_occ(total_counts,arts_of_occurrence, quant=0.333):
    lbl_arts_occ = []
    min_obj = (None,100)
    max_obj = (None,0)
    for label in list(total_counts):
        val = round((total_counts[label]['spans']/total_counts['Overall']['spans'])*100,2)
        lbl_arts_occ.append((label,val))
        if val<min_obj[1]:
            min_obj = (label,val)
        if val>max_obj[1]:
            max_obj = (label,val)
    vals = [entry[1] for entry in lbl_arts_occ]
    thresh = np.quantile(vals, quant)
    minority_labels = []
    for entry in lbl_arts_occ:
        if entry[1]<thresh:
            minority_labels.append(entry[0])
    min_label = min_obj[0]
    max_label = max_obj[0]
    arts_of_interest = []
    for label in minority_labels:
        arts_of_interest.extend(arts_of_occurrence[label])
    arts_of_interest = list(set(arts_of_interest))
    print("Minority Labels:",minority_labels)
    return arts_of_interest, minority_labels, min_label, max_label

def refine_arts_of_int(method, art_lbl_tracking, arts_of_interest, minority_labels, min_label, max_label):
    aoi_dict = dict()
    for key in arts_of_interest:
        aoi_dict[key] = art_lbl_tracking[key]
    if method=="minority":# sort by lowest minority label
        arts_inorder = sorted(aoi_dict, key=lambda x:aoi_dict[x][min_label]['span_pct'], reverse=True)
        #aoi_dict_sorted = {key: aoi_dict[key] for key in arts_inorder}
        overs_arts = arts_inorder[:int(len(arts_inorder)/2)]
    elif method=="dscmin+ascmax":
        arts_dscmin = sorted(aoi_dict, key=lambda x:aoi_dict[x][min_label]['span_pct'], reverse=True)
        arts_ascmax = sorted(aoi_dict, key=lambda x:aoi_dict[x][max_label]['span_pct'], reverse=False)
        #aoi_dict_dscmin = {key: aoi_dict[key] for key in arts_dscmin}
        #aoi_dict_ascmax = {key: aoi_dict[key] for key in arts_ascmax}
        # may need to find a way of reducing the number of articles dynamically?
        # for now just take the first half
        overs_dscmin = arts_dscmin[:int(len(arts_dscmin)/2)]
        overs_ascmax = arts_ascmax[:int(len(arts_ascmax)/2)]
        overs_arts = list(set(overs_dscmin+overs_ascmax))
    elif method=="minorities":
        tple_coll = {label:[] for label in minority_labels}
        art_coll = {label:[] for label in minority_labels}
        overs_arts = []
        for label in minority_labels:
            tplelst = sorted(aoi_dict, key=lambda x:aoi_dict[x][label]['span_pct'], reverse=True)
            tple_coll[label] = tplelst
            art_coll[label] = tplelst[:int(len(tplelst)/len(minority_labels))]
            overs_arts.extend(art_coll[label])
        overs_arts = list(set(overs_arts))
    return overs_arts

def create_new_osds(ds, overs_arts):
    osds = ds.filter(lambda example: example["id"] in overs_arts)
    ovs_ds = datasets.concatenate_datasets([ds, osds])
    return ovs_ds

In [159]:
letter = "a"
htype = "mhead"
ds = datasets.load_from_disk(f"{cwd}/inputs/{letter}/{htype}_ds")
label_lst = get_label_set(letter, htype)
total_counts = get_overall_counts(ds, label_lst)
art_lbl_tracking = article_label_coverage(ds, label_lst, total_counts)
arts_of_occurrence = get_art_lists_for_lbls(art_lbl_tracking, label_lst)
#arts_of_interest, minority_labels, min_label, max_label = compare_art_lbl_occ(arts_of_occurrence, quant=0.3)
arts_of_interest, minority_labels, min_label, max_label = compare_overall_lbl_occ(total_counts,arts_of_occurrence, quant=0.333)
overs_arts = refine_arts_of_int("minorities", art_lbl_tracking, arts_of_interest, minority_labels, min_label, max_label)
osds = create_new_osds(ds, overs_arts)
print("New")
new_counts = get_overall_counts(osds, label_lst)

Actor: 50.26% all spans (5381)
InstrumentType: 29.38% all spans (3145)
Objective: 8.8% all spans (942)
Resource: 4.49% all spans (481)
Time: 7.07% all spans (757)
Overall: 100.0% all spans (10706)
Minority Labels: ['Resource', 'Time']
New
Actor: 49.44% all spans (9454)
InstrumentType: 29.07% all spans (5558)
Objective: 8.87% all spans (1696)
Resource: 5.03% all spans (962)
Time: 7.59% all spans (1451)
Overall: 100.0% all spans (19121)
